# Agent Sandbox
Breaks down `agent.py` step by step so each piece can be inspected and tested independently.

In [ ]:
import sys
sys.path.insert(0, '..')

from dotenv import load_dotenv
load_dotenv('../.env')

## 1. Imports

In [ ]:
import aiosqlite
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import HumanMessage
from langgraph.graph import StateGraph, MessagesState, END
from langgraph.prebuilt import ToolNode
from langgraph.checkpoint.sqlite.aio import AsyncSqliteSaver
from langfuse.langchain import CallbackHandler
from deps import make_obj
from tool import document_search

## 2. Prompt
The system prompt tells the LLM who it is and what it can answer.

In [ ]:
AGENT_PROMPT = ChatPromptTemplate.from_messages([
    ("system", """You are DiabeticAssist, a clinical reference chatbot specializing in diabetes care.
You answer questions using verified medical documents only."""),
    MessagesPlaceholder("messages"),
])

print(AGENT_PROMPT)

## 3. LLM + Tools
`bind_tools` tells the LLM it can call `document_search` when it needs to retrieve context.

In [ ]:
tools = [document_search]
tool_node = ToolNode(tools)
llm = AGENT_PROMPT | make_obj().bind_tools(tools)

print("Tools:", [t.name for t in tools])

## 4. Graph Nodes
`call_model` runs the LLM. `should_continue` decides whether to call a tool or stop.

In [ ]:
def call_model(state: MessagesState):
    response = llm.invoke({"messages": state["messages"]})
    return {"messages": [response]}

def should_continue(state: MessagesState):
    last_message = state["messages"][-1]
    if last_message.tool_calls:
        return "tools"   # LLM wants to call a tool
    return END            # LLM produced a final answer

## 5. Build the Graph
Wires agent → tools → agent in a loop until `should_continue` returns END.

In [ ]:
def build_graph():
    graph = StateGraph(MessagesState)
    graph.add_node("agent", call_model)
    graph.add_node("tools", tool_node)
    graph.set_entry_point("agent")
    graph.add_conditional_edges("agent", should_continue)
    graph.add_edge("tools", "agent")
    return graph

print("Graph built")

## 6. Compile with Checkpointer
The checkpointer saves message history to SQLite so the agent remembers previous turns.

In a notebook we can `await` directly — no need to wrap in `async def`.

In [ ]:
conn = await aiosqlite.connect("sandbox.db")
checkpointer = AsyncSqliteSaver(conn)
await checkpointer.setup()

agent = build_graph().compile(checkpointer=checkpointer)
print("Agent compiled")

## 7. Run the Agent
Send a message and collect the full response.

In [ ]:
langfuse_handler = CallbackHandler()
config = {"configurable": {"thread_id": "sandbox-1"}, "callbacks": [langfuse_handler]}

response = await agent.ainvoke(
    {"messages": [HumanMessage(content="What is the HbA1c target for type 2 diabetes?")]},
    config=config,
)

print(response["messages"][-1].content)